[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S16/2_Practica/para_casa/S16_01_demostrador.ipynb)

# S16 · 01 — Plantilla de demostrador

Sprint 16 · Proyecto: evaluación, IA responsable y entrega (píldora de apoyo)

**Contexto TurisData.** La rúbrica pide un **demostrador ejecutable** (requisito R7) y que sea **reproducible desde el README** (criterio F). Un demostrador no es "el notebook con todo": es una herramienta con la que **otra persona** prueba tu modelo sin saber programar, con **avisos de incertidumbre** y **errores del usuario bien gestionados**. Aquí construimos una plantilla completa con el modelo de cancelaciones de TurisData que tu equipo adaptará a su proyecto.

**Cuatro opciones de demostrador (elige una, la primera es la recomendada):**

| Opción | Cuándo | Ventaja | Inconveniente |
|---|---|---|---|
| **Streamlit** (`app.py`) | Modelos tabulares, series, texto | Web interactiva con pocas líneas | Hay que instalarlo y lanzar `streamlit run` |
| **Gradio** | Imágenes, audio, chat (Hugging Face) | Muy rápido para subir una foto o escribir | Menos control del diseño |
| **Notebook con widgets** | Sin instalar nada / como plan B | Funciona en Colab | Menos "producto" |
| **API con FastAPI** (opcional) | Si otro sistema debe llamar al modelo | Estándar en producción | Requiere cliente que la consuma |

En este cuaderno construimos tres (Streamlit, widgets y FastAPI). Gradio no tiene sección propia: úsalo si tu proyecto trabaja con imágenes, audio o chat.

**Este cuaderno tiene 3 ejercicios**, todos para casa: el cuaderno «en clase» solo trae los ejemplos.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, json, py_compile
from datetime import date
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEMILLA = 42
np.random.seed(SEMILLA)
CARPETA = "demostrador"                    # aquí se crean modelo/, logica.py, app.py y api.py
os.makedirs(os.path.join(CARPETA, "modelo"), exist_ok=True)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Entrenar el modelo de cancelaciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué AUC obtiene el modelo en el test?
2. ¿Qué significa un AUC de 0,5?
3. ¿Por qué el `Pipeline` guarda juntos el preprocesado y el modelo?

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
NUMERICAS = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche", "cliente_repetidor", "cancelaciones_previas"]
CATEGORICAS = ["tipo_habitacion", "canal", "tarifa", "deposito"]
X, y = reservas[NUMERICAS + CATEGORICAS], reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

prep = ColumnTransformer([("num", StandardScaler(), NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
modelo = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)
auc_test = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
print(f"AUC en test: {auc_test:.3f}")

### 👀 A3 · EJEMPLO — Guardar el modelo y los metadatos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué dos archivos se guardan y para qué sirve cada uno?
2. ¿Por qué guardar las versiones de las librerías?
3. ¿Qué campo avisa del uso que NO debe hacerse del modelo?

In [ ]:
meta = {
    "nombre": "Riesgo de cancelación de reservas · TurisData (datos sintéticos)",
    "campos": {
        "antelacion_dias": {"tipo": "entero", "min": int(X["antelacion_dias"].min()), "max": int(X["antelacion_dias"].max()), "etiqueta": "Antelación (días entre reserva y llegada)"},
        "noches": {"tipo": "entero", "min": 1, "max": 30, "etiqueta": "Noches"},
        "adultos": {"tipo": "entero", "min": 1, "max": 6, "etiqueta": "Adultos"},
        "ninos": {"tipo": "entero", "min": 0, "max": 6, "etiqueta": "Niños"},
        "precio_noche": {"tipo": "decimal", "min": 20.0, "max": 600.0, "etiqueta": "Precio por noche (€)"},
        "cliente_repetidor": {"tipo": "entero", "min": 0, "max": 1, "etiqueta": "Cliente repetidor (0 = no, 1 = sí)"},
        "cancelaciones_previas": {"tipo": "entero", "min": 0, "max": 10, "etiqueta": "Cancelaciones previas"},
        **{c: {"tipo": "categoria", "valores": sorted(X[c].unique().tolist()), "etiqueta": c.replace("_", " ").capitalize()} for c in CATEGORICAS},
    },
    "auc_test": round(float(auc_test), 3),
    "n_entrenamiento": int(len(X_ent)),
    "tasa_cancelacion_base": round(float(y.mean()), 3),
    "entrenado": str(date.today()),
    "semilla": SEMILLA,
    "versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "limitaciones": "Entrenado con datos sintéticos de un solo alojamiento ficticio. No usar para decidir sobre personas concretas: solo orienta la gestión de disponibilidad.",
}
joblib.dump(modelo, os.path.join(CARPETA, "modelo", "modelo_cancelaciones.joblib"))
with open(os.path.join(CARPETA, "modelo", "metadatos.json"), "w", encoding="utf8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Guardado:", os.listdir(os.path.join(CARPETA, "modelo")))

### 👀 A4 · EJEMPLO — Escribir logica.py y probarla

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué probabilidad de cancelación y qué nivel sale para `reserva_ejemplo`?
2. ¿Qué devuelve `predecir_reserva` si la entrada es errónea?
3. ¿Por qué `logica.py` no importa Streamlit?

In [ ]:
LOGICA = r'''"""Lógica del demostrador: cargar el modelo, validar la entrada y predecir. Sin interfaz."""
import json
import os

import joblib
import pandas as pd

CARPETA = os.path.dirname(os.path.abspath(__file__))


def cargar_modelo(carpeta=None):
    """Devuelve (modelo, metadatos). Lanza FileNotFoundError con un mensaje claro si faltan los archivos."""
    carpeta = carpeta or os.path.join(CARPETA, "modelo")
    ruta_modelo = os.path.join(carpeta, "modelo_cancelaciones.joblib")
    ruta_meta = os.path.join(carpeta, "metadatos.json")
    if not (os.path.exists(ruta_modelo) and os.path.exists(ruta_meta)):
        raise FileNotFoundError("Falta el modelo. Ejecuta primero el notebook S16_01 (o tu script de entrenamiento) para crear la carpeta 'modelo/'.")
    with open(ruta_meta, encoding="utf8") as f:
        return joblib.load(ruta_modelo), json.load(f)


def validar_reserva(reserva, meta):
    """Devuelve una lista de errores en español (lista vacía = entrada válida)."""
    errores = []
    for campo, regla in meta["campos"].items():
        if campo not in reserva or reserva[campo] in (None, ""):
            errores.append(f"Falta el campo '{campo}'.")
            continue
        valor = reserva[campo]
        if regla["tipo"] == "categoria":
            if valor not in regla["valores"]:
                errores.append(f"'{valor}' no es un valor válido para '{campo}'. Opciones: {', '.join(regla['valores'])}.")
        else:
            try:
                numero = float(valor)
            except (TypeError, ValueError):
                errores.append(f"'{campo}' debe ser un número (has escrito '{valor}').")
                continue
            if not regla["min"] <= numero <= regla["max"]:
                errores.append(f"'{campo}' debe estar entre {regla['min']} y {regla['max']} (has escrito {valor}).")
    return errores


def nivel_riesgo(probabilidad):
    """Traduce la probabilidad a una etiqueta que entiende quien no sabe de modelos."""
    if probabilidad < 0.30:
        return "bajo"
    if probabilidad < 0.50:
        return "medio"
    return "alto"


def predecir_reserva(reserva, modelo, meta):
    """Valida y predice. Devuelve un diccionario; si hay errores, solo {'errores': [...]}."""
    errores = validar_reserva(reserva, meta)
    if errores:
        return {"errores": errores}
    fila = pd.DataFrame([reserva])[list(meta["campos"])]
    probabilidad = float(modelo.predict_proba(fila)[0, 1])
    return {
        "errores": [],
        "probabilidad": round(probabilidad, 3),
        "nivel": nivel_riesgo(probabilidad),
        "tasa_base": meta["tasa_cancelacion_base"],
        "aviso": (f"Estimación orientativa. El modelo tiene un AUC de {meta['auc_test']} en datos de prueba: "
                  "se equivoca con frecuencia. "
                  + meta["limitaciones"]),
    }
'''
with open(os.path.join(CARPETA, "logica.py"), "w", encoding="utf8") as f:
    f.write(LOGICA)

sys.path.insert(0, CARPETA)
import importlib, logica
importlib.reload(logica)
modelo_cargado, meta_cargado = logica.cargar_modelo()

reserva_ejemplo = {"antelacion_dias": 120, "noches": 5, "adultos": 2, "ninos": 1, "precio_noche": 95.0, "cliente_repetidor": 0,
                   "cancelaciones_previas": 0, "tipo_habitacion": "estandar", "canal": "web_intermediaria", "tarifa": "flexible", "deposito": "sin_deposito"}
print(logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado))
print(logica.predecir_reserva({**reserva_ejemplo, "noches": -3, "canal": "telepatia"}, modelo_cargado, meta_cargado))

### ✏️ A5 · EJERCICIO 1 de 3 — Comprobar la coherencia entre campos

**Tu turno:** Las validaciones por campo no detectan **incoherencias entre campos**. Escribe `validar_coherencia(reserva)` que devuelva una lista de errores (vacía si todo va bien) con estas dos reglas: 1. Si hay niños (`ninos > 0`) tiene que haber al menos un adulto (`adultos >= 1`): error `"Los menores deben viajar con al menos un adulto."` 2. Si el importe total (`noches * precio_noche`) supera 5000 €: error que empiece por `"El importe total supera los 5000 €"`. *Nota: en el demostrador el campo `adultos` ya exige un mínimo de 1, así que con la interfaz la regla 1 casi nunca salta; aquí la practicas con una reserva de 0 adultos porque esta función es la red de seguridad entre campos.*

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Por qué hacen falta validaciones entre campos además de las de cada campo?
2. ¿Qué devuelve la función si la reserva es coherente?
3. ¿Pueden salir los dos errores a la vez?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
def validar_coherencia(reserva):
    errores = []
    # TODO: Dos condiciones independientes con if; añade el texto exacto a la lista
    ...
    return errores

assert validar_coherencia(reserva_ejemplo) == [], "La reserva de ejemplo es coherente: la lista debe estar vacía"
assert validar_coherencia({**reserva_ejemplo, "adultos": 0, "ninos": 2}) == ["Los menores deben viajar con al menos un adulto."], "Niños sin adultos: revisa el texto del error"
assert len(err) == 1 and err[0].startswith("El importe total supera los 5000 €"), "Un importe de 8000 € debe dar un error que empiece por 'El importe total supera los 5000 €'"
print("✔ Ejercicio 1 correcto")

### ✏️ A6 · EJERCICIO 2 de 3 — Convertir la lista de errores en un mensaje

**Tu turno:** Escribe `mensaje_errores(errores)`: recibe una lista de errores y devuelve **un único texto** para mostrar al usuario. Debe empezar por `"No se puede calcular todavía:"` y listar cada error en una línea que empiece por `"- "`. Si la lista está vacía, devuelve una cadena vacía `""`.

**Debe salir:** un mensaje con los dos errores de la reserva errónea (cabecera `No se puede calcular todavía:` y una línea `- ...` por error) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Qué devuelve `mensaje_errores([])`?
2. ¿Con qué empieza cada línea de error?
3. ¿Por qué el mensaje dice cómo arreglar el error y no solo que existe?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
def mensaje_errores(errores):
    # TODO: Si no hay errores devuelve "" ; si los hay, une con saltos de línea las líneas "- ..."
    ...

assert mensaje_errores([]) == "", "Sin errores, devuelve una cadena vacía"
m = mensaje_errores(["A", "B"])
assert m.startswith("No se puede calcular todavía:") and "- A" in m and "- B" in m, "Formato: cabecera y una línea '- error' por cada error"
print(mensaje_errores(logica.validar_reserva({**reserva_ejemplo, "noches": -3, "canal": "telepatia"}, meta_cargado)))
print("✔ Ejercicio 2 correcto")

### ✏️ A7 · EJERCICIO 3 de 3 — Dos reservas de comparación

**Tu turno:** Define dos reservas completas (diccionarios con los 11 campos, como `reserva_ejemplo`): `reserva_riesgo_alto` (mucha antelación, sin depósito, tarifa flexible, canal `web_intermediaria`) y `reserva_riesgo_bajo` (poca antelación, depósito total, tarifa no reembolsable, canal `directo`). Calcula sus probabilidades en `p_alto` y `p_bajo` con `logica.predecir_reserva`.

**Debe salir:** `riesgo alto 95% · riesgo bajo 4%` (aproximadamente) y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué probabilidades salen para cada reserva?
2. ¿Qué diferencia mínima se exige entre las dos?
3. ¿Por qué conviene probar el modelo con casos de sentido común?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: copia reserva_ejemplo con {**reserva_ejemplo, "campo": valor} y cambia solo lo necesario; la probabilidad está en la clave "probabilidad"
reserva_riesgo_alto = ...
reserva_riesgo_bajo = ...
p_alto = ...
p_bajo = ...

assert logica.validar_reserva(reserva_riesgo_alto, meta_cargado) == [] and logica.validar_reserva(reserva_riesgo_bajo, meta_cargado) == [], "Ambas reservas deben ser válidas (11 campos, valores permitidos)"
assert 0 <= p_bajo <= 1 and 0 <= p_alto <= 1, "Las probabilidades deben estar entre 0 y 1"
assert p_alto > p_bajo + 0.3, "La reserva de riesgo alto debería superar a la de riesgo bajo en más de 0,30; revisa qué campos has cambiado"
print(f"riesgo alto {p_alto:.0%} · riesgo bajo {p_bajo:.0%}")
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Pruebas de sentido común (las de tests/test_logica.py)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Cuántas pruebas pasan y cuál comprueba el sentido común?
2. ¿Qué comprueba `test_entrada_invalida_no_rompe`?
3. ¿Con qué comando se ejecutarían desde la terminal?

In [ ]:
# Estas pruebas se guardarían tal cual en tests/test_logica.py (se ejecutan con:  pytest -q)
def test_reserva_valida_devuelve_probabilidad():
    r = logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado)
    assert 0 <= r["probabilidad"] <= 1 and r["nivel"] in ("bajo", "medio", "alto") and "aviso" in r

def test_entrada_invalida_no_rompe():
    r = logica.predecir_reserva({**reserva_ejemplo, "noches": -3}, modelo_cargado, meta_cargado)
    assert r["errores"] and "probabilidad" not in r

def test_falta_un_campo():
    incompleta = {k: v for k, v in reserva_ejemplo.items() if k != "canal"}
    assert any("canal" in e for e in logica.validar_reserva(incompleta, meta_cargado))

def test_sentido_comun():
    assert p_alto > p_bajo

for prueba in (test_reserva_valida_devuelve_probabilidad, test_entrada_invalida_no_rompe, test_falta_un_campo, test_sentido_comun):
    prueba()
    print("OK ", prueba.__name__)

### 👀 A9 · EJEMPLO — Generar app.py (Streamlit)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Se ejecuta la app dentro del notebook?
2. ¿Qué muestra la app cuando la entrada es válida?
3. ¿Dónde está la lógica de validar y predecir?

In [ ]:
APP = r'''"""Demostrador Streamlit · TurisData. Lanzar con:  streamlit run app.py"""
import streamlit as st

from logica import cargar_modelo, predecir_reserva

st.set_page_config(page_title="Riesgo de cancelación · TurisData", layout="centered")


@st.cache_resource
def cargar():
    return cargar_modelo()


st.title("Riesgo de cancelación de una reserva")
st.caption("Demostrador del proyecto. Datos sintéticos de TurisData Canarias (empresa ficticia).")

try:
    modelo, meta = cargar()
except FileNotFoundError as error:
    st.error(str(error))
    st.stop()

with st.form("formulario"):
    reserva = {}
    columnas = st.columns(2)
    for i, (campo, regla) in enumerate(meta["campos"].items()):
        with columnas[i % 2]:
            if regla["tipo"] == "categoria":
                reserva[campo] = st.selectbox(regla["etiqueta"], regla["valores"])
            elif regla["tipo"] == "entero":
                reserva[campo] = st.number_input(regla["etiqueta"], value=int(max(regla["min"], min(regla["max"], 1))), step=1)
            else:
                reserva[campo] = st.number_input(regla["etiqueta"], value=float(regla["min"] + 75.0), step=5.0)
    enviado = st.form_submit_button("Calcular riesgo")

if enviado:
    resultado = predecir_reserva(reserva, modelo, meta)
    if resultado["errores"]:
        st.warning("No se puede calcular todavía:\n\n" + "\n".join(f"- {e}" for e in resultado["errores"]))
    else:
        c1, c2 = st.columns(2)
        c1.metric("Probabilidad de cancelación", f"{resultado['probabilidad']:.0%}", f"{resultado['probabilidad'] - resultado['tasa_base']:+.0%} frente a la media")
        c2.metric("Nivel de riesgo", resultado["nivel"].upper())
        st.progress(resultado["probabilidad"])
        st.info(resultado["aviso"])
'''
with open(os.path.join(CARPETA, "app.py"), "w", encoding="utf8") as f:
    f.write(APP)
py_compile.compile(os.path.join(CARPETA, "app.py"), doraise=True)      # comprueba que no tiene errores de sintaxis
print("app.py creado y sin errores de sintaxis. Para verlo: cd demostrador && streamlit run app.py")

### 👀 A10 · EJEMPLO — Formulario con widgets (plan B)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Para qué sirve esta opción?
2. ¿Qué pasa si `ipywidgets` no está instalado?
3. ¿Qué función da el texto cuando hay errores?

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False
    print("ipywidgets no está instalado. En local: pip install ipywidgets (en Colab ya viene incluido).")

if HAY_WIDGETS:
    campos = {}
    for campo, regla in meta_cargado["campos"].items():
        if regla["tipo"] == "categoria":
            campos[campo] = widgets.Dropdown(options=regla["valores"], description=regla["etiqueta"][:22], style={"description_width": "170px"})
        else:
            campos[campo] = widgets.FloatText(value=float(reserva_ejemplo[campo]), description=regla["etiqueta"][:22], style={"description_width": "170px"})
    boton = widgets.Button(description="Calcular riesgo", button_style="primary")
    salida = widgets.Output()

    def al_pulsar(_):
        with salida:
            clear_output()
            reserva = {c: (w.value if isinstance(w, widgets.Dropdown) else (int(w.value) if meta_cargado["campos"][c]["tipo"] == "entero" else w.value))
                       for c, w in campos.items()}
            r = logica.predecir_reserva(reserva, modelo_cargado, meta_cargado)
            if r["errores"]:
                print(mensaje_errores(r["errores"]))
            else:
                print(f"Probabilidad de cancelación: {r['probabilidad']:.0%}  (riesgo {r['nivel']}; media histórica {r['tasa_base']:.0%})")
                print(r["aviso"])
    boton.on_click(al_pulsar)
    display(widgets.VBox(list(campos.values()) + [boton, salida]))

### 👀 A11 · EJEMPLO — Generar api.py (FastAPI)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué dos rutas tiene la API?
2. ¿Qué código devuelve si la entrada es errónea?
3. ¿Es obligatoria la API en el proyecto?

In [ ]:
API = r'''"""API local del modelo. Lanzar con:  uvicorn api:app --reload"""
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from logica import cargar_modelo, predecir_reserva

app = FastAPI(title="Riesgo de cancelación · TurisData", version="1.0")
modelo, meta = cargar_modelo()


class Reserva(BaseModel):
    antelacion_dias: int = Field(ge=0, le=1000)
    noches: int = Field(ge=1, le=30)
    adultos: int = Field(ge=1, le=6)
    ninos: int = Field(ge=0, le=6)
    precio_noche: float = Field(ge=20, le=600)
    cliente_repetidor: int = Field(ge=0, le=1)
    cancelaciones_previas: int = Field(ge=0, le=10)
    tipo_habitacion: str
    canal: str
    tarifa: Literal["flexible", "no_reembolsable"]
    deposito: Literal["sin_deposito", "parcial", "total"]


@app.get("/salud")
def salud():
    return {"estado": "ok", "modelo": meta["nombre"], "auc_test": meta["auc_test"]}


@app.post("/predecir")
def predecir(reserva: Reserva):
    resultado = predecir_reserva(reserva.model_dump(), modelo, meta)
    if resultado["errores"]:
        raise HTTPException(status_code=422, detail=resultado["errores"])
    return resultado
'''
with open(os.path.join(CARPETA, "api.py"), "w", encoding="utf8") as f:
    f.write(API)
py_compile.compile(os.path.join(CARPETA, "api.py"), doraise=True)
print("api.py creado y sin errores de sintaxis.")

### 👀 A12 · EJEMPLO — Comprobar la API (opcional)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué devuelve `/predecir` con `noches = -3`?
2. ¿Qué pasa si no está instalado `fastapi`?

In [ ]:
# Comprobación OPCIONAL de la API (se salta si faltan fastapi o httpx)
try:
    from fastapi.testclient import TestClient
    import api
    importlib.reload(api)
    cliente = TestClient(api.app)
    print("/salud →", cliente.get("/salud").json())
    ok = cliente.post("/predecir", json=reserva_ejemplo)
    print("/predecir (válida) →", ok.status_code, ok.json()["probabilidad"])
    mal = cliente.post("/predecir", json={**reserva_ejemplo, "noches": -3})
    print("/predecir (noches=-3) →", mal.status_code, "(la API rechaza la entrada errónea)")
    mal2 = cliente.post("/predecir", json={**reserva_ejemplo, "canal": "telepatia"})
    print("/predecir (canal inventado) →", mal2.status_code, mal2.json()["detail"][0][:60])
except ImportError as e:
    print("Comprobación de la API omitida (opcional):", e)

### 👀 A13 · EJEMPLO — Generar requirements.txt con las versiones exactas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Por qué se fijan las versiones (`==`)?
2. ¿Qué hace el código con un paquete que no está instalado?

In [ ]:
# Generamos un requirements.txt con las versiones exactas de este entorno (revísalo y recórtalo a lo que uses)
import importlib.metadata as md
paquetes = ["numpy", "pandas", "scikit-learn", "joblib", "streamlit", "ipywidgets", "fastapi", "uvicorn"]
lineas = []
for p in paquetes:
    try:
        lineas.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError:
        lineas.append(f"# {p}  (no instalado aquí: fija la versión que uses)")
with open(os.path.join(CARPETA, "requirements.txt"), "w", encoding="utf8") as f:
    f.write("\n".join(lineas) + "\n")
print("\n".join(lineas))

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** La rúbrica pide un **demostrador ejecutable** (requisito R7) que otra persona pueda lanzar solo con el README (criterio F). Aquí construimos una plantilla completa con el modelo de cancelaciones de TurisData, que tu equipo adaptará a su proyecto.

- **Qué hace:**
    - Define `dato(...)`, que localiza (o descarga) los archivos de datos del curso.
    - Carga `numpy`, `pandas`, `joblib` y de `scikit-learn` lo necesario para entrenar una regresión logística con su preprocesado.
    - Crea la carpeta `demostrador/modelo/`, donde se guardará todo lo que genera este cuaderno.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Guardar el modelo y sus metadatos

#### A2 · Ejemplo — Entrenar el modelo de cancelaciones

**Idea.** El demostrador **no reentrena**: carga un modelo ya entrenado. Guardamos junto al modelo un
archivo `metadatos.json` con todo lo que la interfaz necesita saber: qué campos pedir, qué rangos son
válidos, qué categorías existen, con qué datos y versiones se entrenó y **cuánto vale el modelo**
(así el usuario ve sus límites). El **pipeline** (preprocesado + modelo) se guarda entero con `joblib`,
para que la app reciba una fila "cruda" igual que en el entrenamiento.

**Ejemplo resuelto:** entrenamos una regresión logística con 11 variables sencillas.

- **Qué hace:**
    - Carga las reservas (`reservas_turisdata.csv`) y separa las 7 variables numéricas y las 4 categóricas.
    - Reserva el 20 % para el test (`train_test_split`, con `stratify=y` para mantener la proporción de canceladas).
    - Monta un `Pipeline`: el preprocesado (escalar números y codificar categorías) y la regresión logística, en una sola pieza.
    - Calcula el AUC en el test (`auc_test`).
- **Qué debe salir:**
    - `AUC en test: 0.746` (aproximadamente: puede variar en el tercer decimal según la versión de scikit-learn).
- **Cómo leerlo:**
    - El AUC mide cómo de bien ordena el modelo las reservas: 0,5 es el azar y 1 sería perfecto. **0,75 es un modelo modesto**: acierta bastante más que el azar, pero se equivoca con frecuencia. Por eso más adelante el demostrador lo avisa en cada respuesta.
- **Si falla:**
    - `NameError: name 'dato' is not defined` o `'pd'` → falta la celda de preparación.
    - `FileNotFoundError` / error de descarga de `reservas_turisdata.csv` → comprueba la conexión (la primera vez se descarga).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC obtiene el modelo en el test?* → Alrededor de **0,75** (0,746).
2. *¿Qué significa un AUC de 0,5?* → Que ordena como el **azar**: no aporta nada.
3. *¿Por qué el `Pipeline` guarda juntos el preprocesado y el modelo?* → Para que la app reciba una **fila cruda**, igual que en el entrenamiento, sin repetir el preprocesado a mano.

</details>

#### A3 · Ejemplo — Guardar el modelo y los metadatos

- **Qué hace:**
    - `meta` reúne lo que la interfaz necesita saber: los **campos** a pedir (tipo, mínimo, máximo y etiqueta de cada uno; las categorías con sus valores), el AUC, el tamaño del entrenamiento, la tasa de cancelación base, la fecha, la semilla, las **versiones** de las librerías y las **limitaciones**.
    - `joblib.dump` guarda el pipeline entero en `modelo/modelo_cancelaciones.joblib`.
    - `json.dump` escribe `modelo/metadatos.json` (legible por personas).
- **Qué debe salir:**
    - `Guardado: ['modelo_cancelaciones.joblib', 'metadatos.json']` (los dos archivos, en cualquier orden).
- **Cómo leerlo:**
    - El demostrador **no reentrena**: carga estos dos archivos. Los metadatos permiten que la app construya el formulario y que el usuario vea los **límites** del modelo.
- **Si falla:**
    - `NameError: name 'modelo' is not defined` → falta ejecutar la celda anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos archivos se guardan y para qué sirve cada uno?* → `modelo_cancelaciones.joblib` es el **modelo** (pipeline entero); `metadatos.json` dice qué **campos pedir**, qué **rangos** son válidos y cuáles son los **límites** del modelo.
2. *¿Por qué guardar las versiones de las librerías?* → Para poder **reproducir** el resultado: otra versión puede dar otro modelo.
3. *¿Qué campo avisa del uso que NO debe hacerse del modelo?* → `limitaciones`.

</details>

## 2. La lógica, separada de la interfaz

#### A4 · Ejemplo — Escribir logica.py y probarla

**Idea.** Si validar y predecir viven **dentro** de la app, no se pueden probar ni reutilizar en el
notebook o en la API. Ponemos esa lógica en un archivo `logica.py` **sin nada de Streamlit**. La app,
los widgets y la API serán capas finas que solo llaman a `logica.py`.

**Ejemplo resuelto:** escribimos `logica.py`. Léelo con calma: es lo que adaptará tu equipo.

- **Qué hace:**
    - `LOGICA` es el texto de un archivo `logica.py` con cuatro funciones: `cargar_modelo` (carga o falla con un mensaje claro), `validar_reserva` (lista de errores en español), `nivel_riesgo` (bajo, medio o alto) y `predecir_reserva` (valida, predice y devuelve un diccionario).
    - Se escribe en `demostrador/logica.py`, se **importa** como cualquier módulo y se carga el modelo guardado (`modelo_cargado`, `meta_cargado`).
    - `reserva_ejemplo` es una reserva completa con los 11 campos; se predice con ella y con otra errónea (`noches=-3`, `canal="telepatia"`).
- **Qué debe salir:**
    - Dos diccionarios. El primero: `{'errores': [], 'probabilidad': 0.812, 'nivel': 'alto', 'tasa_base': 0.459, 'aviso': 'Estimación orientativa. El modelo tiene un AUC de 0.746…'}`.
    - El segundo, solo con los errores: `{'errores': ["'noches' debe estar entre 1 y 30 (has escrito -3).", "'telepatia' no es un valor válido para 'canal'. Opciones: agencia, directo, empresa, web_intermediaria."]}`.
- **Cómo leerlo:**
    - **Errores en español y accionables:** dicen qué campo falla, qué se esperaba y qué se recibió.
    - **El resultado incluye el aviso de incertidumbre** (AUC, límites, uso no permitido): viaja con cada respuesta.
    - **Devuelve datos, no imprime:** así la misma función sirve a la app, a los widgets y a la API.
- **Si falla:**
    - `FileNotFoundError: Falta el modelo…` → no has ejecutado la celda que guarda el modelo.
    - `NameError: name 'CARPETA' is not defined` → falta la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué probabilidad de cancelación y qué nivel sale para `reserva_ejemplo`?* → **0,812** (≈ 81 %), nivel **alto** (la media histórica es 0,459).
2. *¿Qué devuelve `predecir_reserva` si la entrada es errónea?* → Solo `{'errores': [...]}`, **sin probabilidad**: no se predice con datos malos.
3. *¿Por qué `logica.py` no importa Streamlit?* → Para poder **probarla y reutilizarla** en la app, los widgets y la API.

</details>

#### A5 · Ejercicio 1 — Comprobar la coherencia entre campos

Las validaciones por campo no detectan **incoherencias entre campos**: cada valor puede ser válido por separado y la combinación no tener sentido. Esta función es la red de seguridad entre campos.

- **Qué hace:**
    - Crea la lista `errores` vacía.
    - Primer `if`: si `ninos > 0` y `adultos < 1`, añade `"Los menores deben viajar con al menos un adulto."`.
    - Segundo `if`: si `noches * precio_noche > 5000`, añade `"El importe total supera los 5000 €: revisa las noches y el precio."`.
    - Devuelve la lista: vacía si todo va bien. Son dos `if` **independientes**: una reserva puede incumplir las dos reglas.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto` (la función no imprime nada).
- **Cómo leerlo:**
    - En la interfaz el campo `adultos` ya exige mínimo 1, así que la regla 1 casi nunca salta; por eso se practica con 0 adultos. Una validación **por capas** (por campo y entre campos) evita que un error del usuario llegue al modelo.
- **Si falla:**
    - `La reserva de ejemplo es coherente: la lista debe estar vacía` → devuelves algún error siempre, o la condición está al revés.
    - `Niños sin adultos: revisa el texto del error` → el texto no coincide exactamente con `"Los menores deben viajar con al menos un adulto."`.
    - `Un importe de 8000 € debe dar un error que empiece por…` → falta la regla 2, o el texto no empieza por `El importe total supera los 5000 €`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué hacen falta validaciones entre campos además de las de cada campo?* → Porque cada valor puede ser válido **por separado** y la **combinación** no tener sentido.
2. *¿Qué devuelve la función si la reserva es coherente?* → Una **lista vacía**.
3. *¿Pueden salir los dos errores a la vez?* → **Sí**: son dos `if` independientes.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def validar_coherencia(reserva):
    errores = []
    if reserva["ninos"] > 0 and reserva["adultos"] < 1:
        errores.append("Los menores deben viajar con al menos un adulto.")
    if reserva["noches"] * reserva["precio_noche"] > 5000:
        errores.append("El importe total supera los 5000 €: revisa las noches y el precio.")
    return errores

err = validar_coherencia({**reserva_ejemplo, "noches": 20, "precio_noche": 400.0})
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejercicio 2 — Convertir la lista de errores en un mensaje

Escribe `mensaje_errores(errores)`: recibe una lista de errores y devuelve **un único texto** para mostrar
al usuario. Debe empezar por `"No se puede calcular todavía:"` y listar cada error en una línea que empiece
por `"- "`. Si la lista está vacía, devuelve una cadena vacía `""`.

- **Qué hace:**
    - Si la lista está vacía, devuelve `""`.
    - Si hay errores, devuelve la cabecera `"No se puede calcular todavía:"`, un salto de línea y los errores unidos con saltos de línea, cada uno precedido de `"- "` (`"\n".join(f"- {e}" for e in errores)`).
    - La comprobación final aplica la función a los errores de una reserva errónea (`noches=-3`, `canal="telepatia"`).
- **Qué debe salir:**
    - ```
No se puede calcular todavía:
- 'noches' debe estar entre 1 y 30 (has escrito -3).
- 'telepatia' no es un valor válido para 'canal'. Opciones: agencia, directo, empresa, web_intermediaria.
```y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Este es el texto que verá una persona sin conocimientos técnicos: dice **qué falla y cómo arreglarlo**. Streamlit y los widgets lo reutilizan.
- **Si falla:**
    - `Sin errores, devuelve una cadena vacía` → devuelves la cabecera aunque no haya errores.
    - `Formato: cabecera y una línea '- error' por cada error` → falta la cabecera exacta o el `"- "` delante de cada error.
    - `NameError: name 'logica' is not defined` → falta el ejemplo de `logica.py`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve `mensaje_errores([])`?* → Una cadena **vacía** `""`.
2. *¿Con qué empieza cada línea de error?* → Con **`- `** (guion y espacio).
3. *¿Por qué el mensaje dice cómo arreglar el error y no solo que existe?* → Porque la persona que usa la app **no sabe programar**: necesita saber qué cambiar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def mensaje_errores(errores):
    if not errores:
        return ""
    return "No se puede calcular todavía:\n" + "\n".join(f"- {e}" for e in errores)

m = mensaje_errores(["A", "B"])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Pruebas de sentido común

#### A7 · Ejercicio 3 — Dos reservas de comparación

**Idea.** Un modelo puede fallar sin dar error: devolver probabilidades absurdas. Antes de enseñarlo,
escribimos **pruebas** (`assert`) sobre casos cuyo resultado esperamos: *"una reserva con riesgo alto sobre
el papel debe puntuar más que una de riesgo bajo"*. Estas pruebas se guardan en el repositorio
(`tests/test_logica.py`) y suman puntos en el criterio F (pruebas).

- **Qué hace:**
    - `{**reserva_ejemplo, "campo": valor, ...}` copia la reserva de ejemplo y cambia solo esos campos.
    - `reserva_riesgo_alto`: mucha antelación, sin depósito, tarifa flexible y canal `web_intermediaria`; `reserva_riesgo_bajo`: poca antelación, depósito total, tarifa no reembolsable y canal `directo`.
    - `logica.predecir_reserva(...)["probabilidad"]` da la probabilidad de cada una (`p_alto` y `p_bajo`).
- **Qué debe salir:**
    - `riesgo alto 95% · riesgo bajo 4%` (aproximadamente) y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - La diferencia es de **más de 0,30**, así que el modelo se comporta con sentido común. Un modelo que puntuara igual a las dos tendría un fallo aunque no diera ningún error.
- **Si falla:**
    - `Ambas reservas deben ser válidas (11 campos, valores permitidos)` → has escrito un valor que no existe (p. ej. `"flexibe"`) o te falta algún campo.
    - `La reserva de riesgo alto debería superar a la de riesgo bajo en más de 0,30…` → revisa qué campos has cambiado (antelación, depósito, tarifa y canal).
    - `NameError: name 'logica' is not defined` → falta el ejemplo de `logica.py`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué probabilidades salen para cada reserva?* → Alrededor de **95 %** (riesgo alto) y **4 %** (riesgo bajo).
2. *¿Qué diferencia mínima se exige entre las dos?* → **Más de 0,30**.
3. *¿Por qué conviene probar el modelo con casos de sentido común?* → Porque un modelo puede **fallar sin dar error**, con probabilidades absurdas.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
reserva_riesgo_alto = {**reserva_ejemplo, "antelacion_dias": 250, "deposito": "sin_deposito", "tarifa": "flexible", "canal": "web_intermediaria"}
reserva_riesgo_bajo = {**reserva_ejemplo, "antelacion_dias": 7, "deposito": "total", "tarifa": "no_reembolsable", "canal": "directo"}
p_alto = logica.predecir_reserva(reserva_riesgo_alto, modelo_cargado, meta_cargado)["probabilidad"]
p_bajo = logica.predecir_reserva(reserva_riesgo_bajo, modelo_cargado, meta_cargado)["probabilidad"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Pruebas de sentido común (las de tests/test_logica.py)

- **Qué hace:**
    - Define cuatro funciones de prueba con `assert`: una reserva válida devuelve una probabilidad entre 0 y 1, un nivel permitido y el aviso; una entrada inválida no rompe y devuelve errores sin probabilidad; si falta un campo, el error lo nombra; y la reserva de riesgo alto puntúa más que la de riesgo bajo.
    - Un bucle las ejecuta una a una e imprime `OK` con el nombre de cada una.
- **Qué debe salir:**
    - ```
OK  test_reserva_valida_devuelve_probabilidad
OK  test_entrada_invalida_no_rompe
OK  test_falta_un_campo
OK  test_sentido_comun
```
- **Cómo leerlo:**
    - Estas pruebas se guardarían tal cual en `tests/test_logica.py` y se lanzarían con `pytest -q`: suman puntos en el criterio F (pruebas). Si una falla, salta un `AssertionError` con el nombre de la prueba.
- **Si falla:**
    - `NameError: name 'p_alto' is not defined` → falta la celda con las dos reservas de comparación.
    - `AssertionError` en una prueba → el modelo o `logica.py` han cambiado: revisa la última modificación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas pruebas pasan y cuál comprueba el sentido común?* → **Cuatro**; la de sentido común es `test_sentido_comun` (`p_alto > p_bajo`).
2. *¿Qué comprueba `test_entrada_invalida_no_rompe`?* → Que una reserva errónea devuelve **errores y no probabilidad**, sin lanzar excepción.
3. *¿Con qué comando se ejecutarían desde la terminal?* → `pytest -q`.

</details>

## 4. Opción A (recomendada): app de Streamlit

#### A9 · Ejemplo — Generar app.py (Streamlit)

**Idea.** Streamlit convierte un script de Python en una web. El script se **vuelve a ejecutar de arriba
abajo cada vez que el usuario toca algo**. Solo necesita la lógica de la sección 2.

**Ejemplo resuelto:** generamos `demostrador/app.py`. **No lo ejecutamos aquí**: se lanza desde una terminal.

```bash
pip install streamlit joblib scikit-learn pandas
cd demostrador
streamlit run app.py
```
En Colab, Streamlit no se abre directamente (hace falta un túnel); por eso en Colab usa la **opción B**
(widgets) y deja Streamlit para el equipo en local o en *Streamlit Community Cloud*.

- **Qué hace:**
    - `APP` es el texto de `demostrador/app.py`: importa Streamlit y `cargar_modelo` y `predecir_reserva` de `logica.py`.
    - Carga el modelo una sola vez (`@st.cache_resource`) y, si faltan los archivos, muestra el error y se detiene.
    - Construye un formulario con un campo por cada entrada de `meta["campos"]` (lista desplegable para las categorías, numérico para el resto).
    - Al pulsar «Calcular riesgo»: si hay errores muestra un aviso con ellos; si no, la probabilidad, el nivel de riesgo, una barra de progreso y el **aviso de incertidumbre**.
    - Se guarda y `py_compile` comprueba que no tiene errores de sintaxis. **No se ejecuta aquí**.
- **Qué debe salir:**
    - `app.py creado y sin errores de sintaxis. Para verlo: cd demostrador && streamlit run app.py`.
- **Cómo leerlo:**
    - La app no tiene lógica propia: es una **capa fina** sobre `logica.py`. Para verla funcionar hay que lanzarla desde una terminal (`streamlit run app.py`); en Colab usa la opción B.
- **Si falla:**
    - `NameError: name 'CARPETA' is not defined` → falta la celda de preparación.
    - `py_compile.PyCompileError` → has modificado el texto de `APP` y hay un error de sintaxis; el mensaje indica la línea.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Se ejecuta la app dentro del notebook?* → **No**: solo se genera y se comprueba la sintaxis; se lanza con `streamlit run app.py` desde una terminal.
2. *¿Qué muestra la app cuando la entrada es válida?* → La **probabilidad**, el **nivel de riesgo**, una barra y el **aviso de incertidumbre**.
3. *¿Dónde está la lógica de validar y predecir?* → En **`logica.py`**; la app solo la llama.

</details>

## 5. Opción B: notebook con widgets (plan B)

#### A10 · Ejemplo — Formulario con widgets (plan B)

Con `ipywidgets` (viene en Colab) se crea un pequeño formulario dentro del propio notebook. Sirve de
**plan B** si falla la app durante la defensa. Si ipywidgets no está instalado, la celda lo avisa y no falla.

- **Qué hace:**
    - Intenta importar `ipywidgets`; si no está instalado, avisa y sigue (no falla).
    - Crea un control por campo (lista desplegable o número) a partir de los metadatos, un botón «Calcular riesgo» y una zona de salida.
    - Al pulsar el botón, valida y predice con `logica`; si hay errores los muestra con `mensaje_errores` (**ejercicio 2**); si no, la probabilidad, el nivel y el aviso.
- **Qué debe salir:**
    - Con `ipywidgets` (en Colab ya viene): un formulario con los 11 campos y el botón; al pulsarlo aparece, por ejemplo, `Probabilidad de cancelación: 81%  (riesgo alto; media histórica 46%)` y el aviso.
    - Sin él: `ipywidgets no está instalado. En local: pip install ipywidgets (en Colab ya viene incluido).`
- **Cómo leerlo:**
    - Es el **plan B** si falla la app durante la defensa: no instala nada y funciona dentro del notebook. Prueba a poner `noches = -3` y fíjate en el mensaje de error.
- **Si falla:**
    - `NameError: name 'mensaje_errores' is not defined` → falta el ejercicio 2.
    - El botón no hace nada → ejecuta la celda otra vez y pulsa sobre el botón nuevo.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Para qué sirve esta opción?* → Es el **plan B**: un formulario dentro del notebook, sin instalar nada.
2. *¿Qué pasa si `ipywidgets` no está instalado?* → La celda **lo avisa y no falla**.
3. *¿Qué función da el texto cuando hay errores?* → `mensaje_errores`, del **ejercicio 2**.

</details>

## 6. Opción C (opcional): API con FastAPI

#### A11 · Ejemplo — Generar api.py (FastAPI)

**Solo si tu proyecto lo necesita** (p. ej. otro sistema consulta tu modelo). La API expone la misma lógica
en dos rutas: `/salud` (¿está viva?) y `/predecir` (recibe una reserva en JSON). FastAPI valida los tipos
y rangos automáticamente y genera documentación interactiva en `/docs`.

```bash
pip install fastapi uvicorn
cd demostrador
uvicorn api:app --reload        # después abre http://127.0.0.1:8000/docs
```
**OPCIONAL.** Si `fastapi` no está instalado (o no hay `httpx`), la comprobación de abajo se salta.

- **Qué hace:**
    - `API` es el texto de `demostrador/api.py`: define una clase `Reserva` con los 11 campos, sus tipos y rangos (FastAPI valida automáticamente) y dos rutas: `GET /salud` y `POST /predecir`.
    - Si la predicción tiene errores, responde con el código HTTP **422** y la lista de errores.
    - Se guarda y `py_compile` comprueba la sintaxis.
- **Qué debe salir:**
    - `api.py creado y sin errores de sintaxis.`
- **Cómo leerlo:**
    - **Solo si tu proyecto lo necesita** (otro sistema consulta tu modelo). La API reutiliza la misma `logica.py`; al lanzarla con `uvicorn api:app --reload`, `/docs` ofrece una página interactiva.
- **Si falla:**
    - `py_compile.PyCompileError` → has modificado el texto de `API` y hay un error de sintaxis.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos rutas tiene la API?* → `/salud` (¿está viva?) y `/predecir` (recibe una reserva en JSON).
2. *¿Qué código devuelve si la entrada es errónea?* → **422**, con la lista de errores.
3. *¿Es obligatoria la API en el proyecto?* → **No**: es opcional.

</details>

#### A12 · Ejemplo — Comprobar la API (opcional)

- **Qué hace:**
    - Si `fastapi` y `httpx` están instalados, crea un cliente de prueba (`TestClient`) y llama a `/salud`, a `/predecir` con una reserva válida y con dos erróneas (`noches=-3` y un canal inventado).
    - Si falta alguna librería, imprime que la comprobación se omite.
- **Qué debe salir:**
    - Con las librerías: `/salud → {'estado': 'ok', 'modelo': 'Riesgo de cancelación de reservas · TurisData (datos sintéticos)', 'auc_test': 0.746}`, `/predecir (válida) → 200 0.812`, `/predecir (noches=-3) → 422 (la API rechaza la entrada errónea)` y `/predecir (canal inventado) → 422 'telepatia' no es un valor válido para 'canal'. Opciones: ag`.
    - Sin ellas: `Comprobación de la API omitida (opcional): …`.
- **Cómo leerlo:**
    - La API **rechaza** las entradas erróneas con 422 en lugar de devolver una predicción absurda.
- **Si falla:**
    - `NameError: name 'reserva_ejemplo' is not defined` → falta el ejemplo de `logica.py`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve `/predecir` con `noches = -3`?* → El código **422**: la API rechaza la entrada errónea.
2. *¿Qué pasa si no está instalado `fastapi`?* → La comprobación **se omite** (es opcional).

</details>

## 7. Lista de reproducibilidad del demostrador

#### A13 · Ejemplo — Generar requirements.txt con las versiones exactas

Un compañero de otro equipo debería poder ejecutarlo **solo con el README**. Comprueba:
- [ ] `requirements.txt` con **versiones fijas** (`pip freeze` de las librerías que usas, no de todo el sistema).
- [ ] El modelo se crea con un **script o notebook** (`entrenar.py`) o se incluye en el repo (si pesa poco).
- [ ] El README dice **exactamente** qué comandos ejecutar y qué se verá.
- [ ] La app **no se rompe** con entradas erróneas y muestra el aviso de incertidumbre.
- [ ] Hay **pruebas** (`pytest -q`) y **plan B**: vídeo de 90 s de la demo funcionando.
- [ ] No hay claves de API ni datos personales en el repositorio.

- **Qué hace:**
    - Para cada paquete de la lista consulta la versión instalada (`importlib.metadata`) y escribe `paquete==versión`.
    - Si un paquete no está instalado, escribe una línea comentada para recordar que hay que fijar su versión.
    - Guarda el resultado en `demostrador/requirements.txt` y lo muestra.
- **Qué debe salir:**
    - Una línea por paquete, por ejemplo `scikit-learn==1.9.1` (las versiones de tu entorno) y, para los que no estén, `# streamlit  (no instalado aquí: fija la versión que uses)`.
- **Cómo leerlo:**
    - Un `requirements.txt` con **versiones fijas** es lo que permite a otra persona reproducir tu demostrador. Revísalo y recórtalo a lo que realmente uses.
- **Si falla:**
    - `NameError: name 'CARPETA' is not defined` → falta la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se fijan las versiones (`==`)?* → Para que otra persona **instale lo mismo** y obtenga el mismo resultado.
2. *¿Qué hace el código con un paquete que no está instalado?* → Escribe una **línea comentada** que recuerda fijar su versión.

</details>

## Mini-reto para tu proyecto · 🏠 en casa

Adapta la plantilla a **tu proyecto**: (1) guarda tu mejor modelo y sus metadatos; (2) modifica `logica.py`
(campos, validaciones y aviso de incertidumbre propios); (3) lanza `app.py`; (4) escribe 3 pruebas;
(5) graba un vídeo de 90 segundos como plan B. Comprueba que otra persona del equipo lo ejecuta desde cero.

*(Este mini-reto se hace en el repositorio de tu equipo, no aquí: no se autocorrige.)*

## Preguntas de reflexión
1. ¿Por qué es mejor que `logica.py` no importe Streamlit? ¿Qué ganamos al reutilizarla en la API y en el notebook?
2. El aviso de incertidumbre viaja en cada respuesta. ¿Qué otro aviso añadirías para tu proyecto (imágenes, texto, series)?
3. ¿Qué pasa si el usuario introduce un valor fuera del rango de entrenamiento y la validación lo permite? ¿Cómo lo detectarías?
4. ¿Qué datos personales podría recibir tu demostrador y cómo evitas guardarlos?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque así la lógica se puede **probar sin abrir ninguna interfaz** y se reutiliza igual en la app, los widgets y la API. Si cambias de interfaz solo cambias una capa fina y no duplicas validaciones.
2. Depende del proyecto. Imágenes: «solo reconoce lo parecido a las fotos de entrenamiento (luz, fondo, tipo de objeto)»; texto: idioma y longitud con los que se entrenó; series: horizonte de predicción fiable. Siempre: para qué **no** debe usarse.
3. El modelo devuelve una probabilidad **sin avisar** de que está extrapolando. Se detecta guardando en los metadatos el rango visto en el entrenamiento de cada variable y avisando cuando un valor queda cerca o fuera de él.
4. Por ejemplo, nombre, correo o documento de identidad si el formulario los pidiera. Se evita **no pidiéndolos**, no guardando las entradas en registros (logs) y, si hay que guardarlas, anonimizándolas y avisando de la privacidad. Nunca claves ni datos personales en el repositorio.

</details>

## Resumen: qué has aprendido
- Guardar modelo + metadatos y cargarlos desde otra capa.
- Separar lógica e interfaz; validar entradas y mostrar incertidumbre.
- Escribir pruebas de sentido común para un modelo.
- Tres formas de demostrador (Streamlit, widgets, API) y el plan B.